# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset

### In Your Response:
1. What does the dataset include?
2. How many rows and columns are present?


In [ ]:
# Add code here 🔧

In [3]:
file_path = '/content/cleaned_airbnb_data.csv' # Using cleaned_airbnb_data.csv based on available files
df = pd.read_csv(file_path, engine='python')
display(df.head())

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,11156,https://www.airbnb.com/rooms/11156,20250610032053,2025-06-17,city scrape,An Oasis in the City,Very central to the city which can be reached ...,"It is very close to everything and everywhere,...",https://a0.muscache.com/pictures/2797669/17895...,40855,...,4.76,4.82,4.71,NaN,f,1,0,1,0,1.02
1,15253,https://www.airbnb.com/rooms/15253,20250610032053,2025-06-17,city scrape,Unique Designer Rooftop Apartment in City Loca...,You will be staying in a unique apartment on t...,The location is really central and there is nu...,https://a0.muscache.com/pictures/a41641fb-0e5a...,59850,...,4.75,4.76,4.56,PID-STRA-24061-7,t,1,0,1,0,3.81
2,44545,https://www.airbnb.com/rooms/44545,20250610032053,2025-06-17,city scrape,Sunny Darlinghurst Warehouse Apartment,Sunny warehouse/loft apartment in the heart of...,Darlinghurst is home to some of Sydney's best ...,https://a0.muscache.com/pictures/a88d8e14-4f63...,112237,...,4.96,4.93,4.79,PID-STRA-74219,f,1,1,0,0,0.46
3,58506,https://www.airbnb.com/rooms/58506,20250610032053,2025-06-17,city scrape,"Studio Yindi @ Mosman, Sydney","An open plan apartment, adjacent to a spacious...","Mosman is a smart, middle to upper class subur...",https://a0.muscache.com/pictures/23497720/d30f...,279955,...,4.92,4.76,4.73,PID-STRA-2810,f,1,1,0,0,2.54
4,68999,https://www.airbnb.com/rooms/68999,20250610032053,2025-06-17,city scrape,A little bit of Sydney - Australia,"Hello Everyone,<br /><br />We have a quiet are...",NaN,https://a0.muscache.com/pictures/5264473/5bec1...,333581,...,4.99,4.85,4.93,PID-STRA-9081,f,1,0,1,0,0.70


### ✍️ Your Response: 🔧

1. The dataset includes detailed information about Airbnb listings, covering host attributes, listing details, location data, property characteristics, and calendar/booking variables.
2. The dataset has 18187 rows and 77 columns.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

### In Your Response:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?


In [4]:
columns_to_drop = ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleared', 'neighbourhood_group_cleared', 'zipcode', 'license', 'calendar_last_scraped', 'first_review', 'last_review']

df = df.drop(columns=columns_to_drop, errors='ignore')

print("Remaining columns after dropping:")
print(df.columns)

Remaining columns after dropping:
Index(['host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_listings_count', 'host_total_listings_count',
       'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type',
       'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms',
       'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights',
       'minimum_minimum_nights', 'maximum_minimum_nights',
       'minimum_maximum_nights', 'maximum_maximum_nights',
       'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'has_availability',
       'availability_30', 'availability_60', 'availability_90',
       'availability_365', 'number_of_reviews', 'number_of_reviews_ltm',
       'number_of_reviews_l30d', 'availability_eoy', 'number_of_reviews_ly',
       'estimated_occupancy_l365d', 'estimated_revenue_l365d',
       'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_ch

### ✍️ Your Response: 🔧

1. I dropped columns that are not useful for predicting price, such as identifiers, URLs, text descriptions, and some date-related fields.
2. Including these columns could introduce noise, increase complexity, require extensive feature engineering, and potentially lead to overfitting, negatively impacting the model's performance on new data.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

### In Your Response:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?


In [6]:
correlation_matrix = df.corr(numeric_only=True)
display(correlation_matrix)

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
host_listings_count,1.000000,0.934186,-0.016291,0.034221,-0.003680,0.015448,-0.018353,-0.033923,0.007752,-0.050826,...,-0.113616,-0.113062,-0.151161,-0.060851,-0.141056,0.762015,0.765530,0.126348,-0.004311,-0.039726
host_total_listings_count,0.934186,1.000000,-0.016498,0.045820,0.019543,0.038215,-0.005947,-0.020799,0.028966,-0.055713,...,-0.140589,-0.130106,-0.171048,-0.070277,-0.172917,0.775040,0.766086,0.155411,-0.006903,-0.057881
latitude,-0.016291,-0.016498,1.000000,0.088195,0.158957,0.119724,0.164784,0.151201,0.112886,0.000242,...,0.036247,0.040379,0.038656,0.038270,0.021332,-0.021244,0.000323,-0.049641,0.015745,-0.059981
longitude,0.034221,0.045820,0.088195,1.000000,-0.005099,0.006068,-0.016148,0.006278,0.155675,0.024029,...,0.088661,0.093692,0.089654,0.232295,0.062461,0.041216,0.106207,-0.132641,0.030698,-0.039312
accommodates,-0.003680,0.019543,0.158957,-0.005099,1.000000,0.675958,0.872330,0.844415,0.364708,-0.056164,...,0.037529,0.053019,0.043026,0.068722,0.015926,0.037151,0.106943,-0.140149,-0.037690,-0.033984
bathrooms,0.015448,0.038215,0.119724,0.006068,0.675958,1.000000,0.718165,0.620272,0.450705,0.005934,...,-0.011526,0.017100,-0.002314,0.021478,0.000241,0.089344,0.094310,0.007986,-0.023905,-0.130313
bedrooms,-0.018353,-0.005947,0.164784,-0.016148,0.872330,0.718165,1.000000,0.787239,0.376457,0.010699,...,0.032401,0.064432,0.045663,0.055329,0.022519,0.027660,0.081906,-0.110178,-0.031729,-0.143068
beds,-0.033923,-0.020799,0.151201,0.006278,0.844415,0.620272,0.787239,1.000000,0.355792,-0.030126,...,0.042639,0.065253,0.054518,0.067099,0.026094,-0.012235,0.044218,-0.115272,-0.009768,-0.045245
price,0.007752,0.028966,0.112886,0.155675,0.364708,0.450705,0.376457,0.355792,1.000000,0.036909,...,0.087119,0.073131,0.061403,0.112290,0.045622,0.028191,0.064195,-0.070991,-0.018273,-0.132156
minimum_nights,-0.050826,-0.055713,0.000242,0.024029,-0.056164,0.005934,0.010699,-0.030126,0.036909,1.000000,...,-0.043472,-0.000222,-0.006809,-0.008929,-0.015717,-0.057054,-0.044940,-0.034625,-0.018428,-0.275369


### ✍️ Your Response: 🔧

1. The variables with the strongest positive correlation with price are `bathrooms`, `bedrooms`, `accommodates`, and `beds`. The variables with the strongest negative correlation are `estimated_occupancy_l365d`, `reviews_per_month`, `number_of_reviews_ltm`, and `number_of_reviews_ly`.

2. Variables that might be useful predictors include those with stronger correlations (both positive and negative) with price, such as `bathrooms`, `bedrooms`, `accommodates`, `beds`, `longitude`, `latitude`, `review_scores_location`, `review_scores_rating`, and the various review count related features (though their negative correlation might need further investigation).

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

### In Your Response:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?


In [8]:
# Define target variable
target = df['price']

# Define features (remove 'price' from the DataFrame)
features = df.drop('price', axis=1)

print("Target variable shape:", target.shape)
print("Features shape:", features.shape)

Target variable shape: (18187,)
Features shape: (18187, 50)


### ✍️ Your Response: 🔧

1. The features I am using are all the columns in the DataFrame except for 'price'.
2. This is a regression problem because we are predicting a continuous numerical value (price), not a category.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [9]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Training features shape: (14549, 50)
Testing features shape: (3638, 50)
Training target shape: (14549,)
Testing target shape: (3638,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [ ]:
# Add code here 🔧

## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

### In Your Response:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?


In [ ]:
# Add code here 🔧

### ✍️ Your Response: 🔧
1.

2.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

### In Your Response:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?


In [ ]:
# Add code here 🔧

### ✍️ Your Response: 🔧
1.

2.

3.


## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

### In Your Response:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?


In [ ]:
# Add code here 🔧

### ✍️ Your Response: 🔧
1.

2.

3.

4.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### In Your Response:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1.

2.

3.

4.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_11_LastnameFirstname.ipynb"